# DQN on Atari Pong — Baseline (Project 1, Step 1)

Ported from **Deep Reinforcement Learning Hands-On**, Chapter 6
(https://github.com/PacktPublishing/Deep-Reinforcement-Learning-Hands-On/tree/master/Chapter06),
updated to run on current library versions:

| Book (2019) | This notebook |
|---|---|
| `gym==0.10.9` | `gymnasium` (Farama fork) |
| `atari-py==0.1.6` | `ale-py` |
| `PongNoFrameskip-v4` (old gym registry) | `ALE/Pong-v5` with `frameskip=1, repeat_action_probability=0.0` (same deterministic, no-internal-skip behavior) |
| `torch==0.4.1`, `ByteTensor` done-masks | current `torch`, boolean masks |
| `tensorboardX` | `torch.utils.tensorboard` (built into PyTorch now) |

The 4-frame skip/stack, 84x84 grayscale preprocessing, replay buffer, target network,
and epsilon-greedy schedule are otherwise **unchanged from the book** — this is meant to
reproduce the textbook's baseline (claimed: mean reward ~19, ~10 min on a GTX 1080 Ti),
not improve on it. Steps 2 (alternative exploration/training algorithms) and 3 (PER)
build on top of this notebook separately.

**Runtime:** Set `Runtime > Change runtime type > A100 GPU` before running.


## 1. Check GPU and record it (you need this number for the report)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv


## 2. Install dependencies

Colab already ships a CUDA-matched PyTorch build, so we leave `torch` alone and only
install the RL/env stack. `ale-py` bundles the Atari ROMs directly (no separate
AutoROM/license step needed with current versions).

In [ ]:
!pip install -q gymnasium ale-py opencv-python-headless

import gymnasium, ale_py, cv2, torch
print("gymnasium", gymnasium.__version__)
print("ale_py   ", ale_py.__version__)
print("cv2      ", cv2.__version__)
print("torch    ", torch.__version__, "| cuda available:", torch.cuda.is_available())


## 3. `lib/wrappers.py` — Atari preprocessing (Chapter 6, ported to Gymnasium API)

Same wrapper stack as the book (`MaxAndSkip` -> `FireReset` -> `ProcessFrame84` ->
`ImageToPyTorch` -> `BufferWrapper(4)` -> `ScaledFloatFrame`), rewritten for
Gymnasium's `(obs, info)` reset and `(obs, reward, terminated, truncated, info)` step.

In [ ]:
import os
os.makedirs("lib", exist_ok=True)


In [ ]:
%%writefile lib/wrappers.py
"""
Atari preprocessing wrappers, ported from
Deep Reinforcement Learning Hands-On, Chapter 6 (lib/wrappers.py),
to the Gymnasium API (terminated/truncated, reset -> (obs, info)).
"""
import collections

import ale_py
import cv2
import gymnasium as gym
import numpy as np

gym.register_envs(ale_py)


class FireResetEnv(gym.Wrapper):
    def __init__(self, env=None):
        super(FireResetEnv, self).__init__(env)
        assert env.unwrapped.get_action_meanings()[1] == 'FIRE'
        assert len(env.unwrapped.get_action_meanings()) >= 3

    def step(self, action):
        return self.env.step(action)

    def reset(self, **kwargs):
        self.env.reset(**kwargs)
        obs, _, terminated, truncated, info = self.env.step(1)
        if terminated or truncated:
            self.env.reset(**kwargs)
        obs, _, terminated, truncated, info = self.env.step(2)
        if terminated or truncated:
            self.env.reset(**kwargs)
        return obs, info


class MaxAndSkipEnv(gym.Wrapper):
    def __init__(self, env=None, skip=4):
        """Return only every `skip`-th frame"""
        super(MaxAndSkipEnv, self).__init__(env)
        self._obs_buffer = collections.deque(maxlen=2)
        self._skip = skip

    def step(self, action):
        total_reward = 0.0
        terminated = truncated = False
        info = {}
        for _ in range(self._skip):
            obs, reward, terminated, truncated, info = self.env.step(action)
            self._obs_buffer.append(obs)
            total_reward += reward
            if terminated or truncated:
                break
        max_frame = np.max(np.stack(self._obs_buffer), axis=0)
        return max_frame, total_reward, terminated, truncated, info

    def reset(self, **kwargs):
        self._obs_buffer.clear()
        obs, info = self.env.reset(**kwargs)
        self._obs_buffer.append(obs)
        return obs, info


class ProcessFrame84(gym.ObservationWrapper):
    def __init__(self, env=None):
        super(ProcessFrame84, self).__init__(env)
        self.observation_space = gym.spaces.Box(low=0, high=255, shape=(84, 84, 1), dtype=np.uint8)

    def observation(self, obs):
        return ProcessFrame84.process(obs)

    @staticmethod
    def process(frame):
        if frame.size == 210 * 160 * 3:
            img = np.reshape(frame, [210, 160, 3]).astype(np.float32)
        elif frame.size == 250 * 160 * 3:
            img = np.reshape(frame, [250, 160, 3]).astype(np.float32)
        else:
            assert False, "Unknown resolution."
        img = img[:, :, 0] * 0.299 + img[:, :, 1] * 0.587 + img[:, :, 2] * 0.114
        resized_screen = cv2.resize(img, (84, 110), interpolation=cv2.INTER_AREA)
        x_t = resized_screen[18:102, :]
        x_t = np.reshape(x_t, [84, 84, 1])
        return x_t.astype(np.uint8)


class ImageToPyTorch(gym.ObservationWrapper):
    def __init__(self, env):
        super(ImageToPyTorch, self).__init__(env)
        old_shape = self.observation_space.shape
        self.observation_space = gym.spaces.Box(low=0.0, high=1.0,
                                                  shape=(old_shape[-1], old_shape[0], old_shape[1]),
                                                  dtype=np.float32)

    def observation(self, observation):
        return np.moveaxis(observation, 2, 0)


class ScaledFloatFrame(gym.ObservationWrapper):
    def observation(self, obs):
        return np.array(obs).astype(np.float32) / 255.0


class BufferWrapper(gym.ObservationWrapper):
    def __init__(self, env, n_steps, dtype=np.float32):
        super(BufferWrapper, self).__init__(env)
        self.dtype = dtype
        old_space = env.observation_space
        self.observation_space = gym.spaces.Box(old_space.low.repeat(n_steps, axis=0),
                                                  old_space.high.repeat(n_steps, axis=0), dtype=dtype)

    def reset(self, **kwargs):
        self.buffer = np.zeros_like(self.observation_space.low, dtype=self.dtype)
        obs, info = self.env.reset(**kwargs)
        return self.observation(obs), info

    def observation(self, observation):
        self.buffer[:-1] = self.buffer[1:]
        self.buffer[-1] = observation
        return self.buffer


def make_env(env_name="ALE/Pong-v5"):
    env = gym.make(env_name, frameskip=1, repeat_action_probability=0.0)
    env = MaxAndSkipEnv(env)
    env = FireResetEnv(env)
    env = ProcessFrame84(env)
    env = ImageToPyTorch(env)
    env = BufferWrapper(env, 4)
    return ScaledFloatFrame(env)


## 4. `lib/dqn_model.py` — the conv net (unchanged from the book, plain PyTorch)

In [ ]:
%%writefile lib/dqn_model.py
import numpy as np
import torch
import torch.nn as nn


class DQN(nn.Module):
    def __init__(self, input_shape, n_actions):
        super(DQN, self).__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(input_shape[0], 32, kernel_size=8, stride=4),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1),
            nn.ReLU()
        )

        conv_out_size = self._get_conv_out(input_shape)
        self.fc = nn.Sequential(
            nn.Linear(conv_out_size, 512),
            nn.ReLU(),
            nn.Linear(512, n_actions)
        )

    def _get_conv_out(self, shape):
        o = self.conv(torch.zeros(1, *shape))
        return int(np.prod(o.size()))

    def forward(self, x):
        conv_out = self.conv(x).view(x.size()[0], -1)
        return self.fc(conv_out)


In [ ]:
%%writefile lib/__init__.py
# marks lib/ as a package


## 5. Training code

Same algorithm as `Chapter06/02_dqn_pong.py`: uniform replay buffer, target network
synced every `SYNC_TARGET_FRAMES`, linear epsilon-greedy decay. Wrapped in a
`run_training()` function (instead of the book's flat `if __name__ == "__main__"`
script) so we can call it once as a quick smoke test and again for the real run,
and so it returns a results dict you can drop straight into the report.

In [ ]:
import collections
import json
import time

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.tensorboard import SummaryWriter

from lib import dqn_model, wrappers

DEFAULT_ENV_NAME = "ALE/Pong-v5"
MEAN_REWARD_BOUND = 19.5

GAMMA = 0.99
BATCH_SIZE = 32
REPLAY_SIZE = 10000
LEARNING_RATE = 1e-4
SYNC_TARGET_FRAMES = 1000
REPLAY_START_SIZE = 10000

EPSILON_DECAY_LAST_FRAME = 10 ** 5
EPSILON_START = 1.0
EPSILON_FINAL = 0.02

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Experience = collections.namedtuple(
    "Experience", field_names=["state", "action", "reward", "done", "new_state"]
)


class ExperienceBuffer:
    def __init__(self, capacity):
        self.buffer = collections.deque(maxlen=capacity)

    def __len__(self):
        return len(self.buffer)

    def append(self, experience):
        self.buffer.append(experience)

    def sample(self, batch_size):
        indices = np.random.choice(len(self.buffer), batch_size, replace=False)
        states, actions, rewards, dones, next_states = zip(*[self.buffer[idx] for idx in indices])
        return (np.array(states), np.array(actions), np.array(rewards, dtype=np.float32),
                np.array(dones, dtype=bool), np.array(next_states))


class Agent:
    def __init__(self, env, exp_buffer):
        self.env = env
        self.exp_buffer = exp_buffer
        self._reset()

    def _reset(self):
        self.state, _ = self.env.reset()
        self.total_reward = 0.0

    @torch.no_grad()
    def play_step(self, net, epsilon=0.0, device="cpu"):
        done_reward = None

        if np.random.random() < epsilon:
            action = self.env.action_space.sample()
        else:
            state_v = torch.as_tensor(np.array([self.state]), dtype=torch.float32).to(device)
            q_vals_v = net(state_v)
            action = int(torch.argmax(q_vals_v, dim=1).item())

        new_state, reward, terminated, truncated, _ = self.env.step(action)
        is_done = terminated or truncated
        self.total_reward += reward

        exp = Experience(self.state, action, reward, is_done, new_state)
        self.exp_buffer.append(exp)
        self.state = new_state
        if is_done:
            done_reward = self.total_reward
            self._reset()
        return done_reward


def calc_loss(batch, net, tgt_net, device="cpu"):
    states, actions, rewards, dones, next_states = batch

    states_v = torch.as_tensor(states, dtype=torch.float32).to(device)
    next_states_v = torch.as_tensor(next_states, dtype=torch.float32).to(device)
    actions_v = torch.as_tensor(actions, dtype=torch.int64).to(device)
    rewards_v = torch.as_tensor(rewards, dtype=torch.float32).to(device)
    done_mask = torch.as_tensor(dones, dtype=torch.bool).to(device)

    state_action_values = net(states_v).gather(1, actions_v.unsqueeze(-1)).squeeze(-1)
    with torch.no_grad():
        next_state_values = tgt_net(next_states_v).max(1)[0]
        next_state_values[done_mask] = 0.0

    expected_state_action_values = next_state_values * GAMMA + rewards_v
    return nn.MSELoss()(state_action_values, expected_state_action_values)


In [ ]:
def run_training(env_name=DEFAULT_ENV_NAME, reward_bound=MEAN_REWARD_BOUND, run_tag="baseline",
                  max_frames=None, replay_size=REPLAY_SIZE, replay_start_size=REPLAY_START_SIZE,
                  sync_target_frames=SYNC_TARGET_FRAMES, epsilon_decay_last_frame=EPSILON_DECAY_LAST_FRAME):
    """Runs the book's baseline DQN training loop until `reward_bound` is hit
    (or `max_frames` is reached, for smoke testing) and returns a results dict."""
    env = wrappers.make_env(env_name)
    net = dqn_model.DQN(env.observation_space.shape, env.action_space.n).to(device)
    tgt_net = dqn_model.DQN(env.observation_space.shape, env.action_space.n).to(device)
    writer = SummaryWriter(log_dir=f"runs/{run_tag}")

    buffer = ExperienceBuffer(replay_size)
    agent = Agent(env, buffer)
    epsilon = EPSILON_START

    optimizer = optim.Adam(net.parameters(), lr=LEARNING_RATE)
    total_rewards = []
    frame_idx = 0
    ts_frame = 0
    ts = time.time()
    start_time = time.time()
    best_mean_reward = None
    solved = False

    while True:
        frame_idx += 1
        epsilon = max(EPSILON_FINAL, EPSILON_START - frame_idx / epsilon_decay_last_frame)

        reward = agent.play_step(net, epsilon, device=device)
        if reward is not None:
            total_rewards.append(reward)
            speed = (frame_idx - ts_frame) / (time.time() - ts)
            ts_frame = frame_idx
            ts = time.time()
            mean_reward = float(np.mean(total_rewards[-100:]))
            print(f"{frame_idx}: done {len(total_rewards)} games, mean reward {mean_reward:.3f}, "
                  f"eps {epsilon:.2f}, speed {speed:.2f} f/s")
            writer.add_scalar("epsilon", epsilon, frame_idx)
            writer.add_scalar("speed", speed, frame_idx)
            writer.add_scalar("reward_100", mean_reward, frame_idx)
            writer.add_scalar("reward", reward, frame_idx)
            if best_mean_reward is None or best_mean_reward < mean_reward:
                torch.save(net.state_dict(), f"{run_tag}-best.dat")
                if best_mean_reward is not None:
                    print(f"Best mean reward updated {best_mean_reward:.3f} -> {mean_reward:.3f}, model saved")
                best_mean_reward = mean_reward
            if mean_reward > reward_bound:
                print(f"Solved in {frame_idx} frames!")
                solved = True
                break

        if max_frames is not None and frame_idx >= max_frames:
            print(f"Stopping at max_frames={max_frames} (smoke test / time-capped run)")
            break

        if len(buffer) < replay_start_size:
            continue

        if frame_idx % sync_target_frames == 0:
            tgt_net.load_state_dict(net.state_dict())

        optimizer.zero_grad()
        batch = buffer.sample(BATCH_SIZE)
        loss_t = calc_loss(batch, net, tgt_net, device=device)
        loss_t.backward()
        optimizer.step()

    writer.close()
    env.close()
    elapsed = time.time() - start_time

    results = {
        "run_tag": run_tag,
        "env_name": env_name,
        "solved": solved,
        "frames": frame_idx,
        "episodes": len(total_rewards),
        "best_mean_reward": best_mean_reward,
        "wall_clock_seconds": elapsed,
        "wall_clock_minutes": elapsed / 60,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    }
    with open(f"{run_tag}-results.json", "w") as f:
        json.dump(results, f, indent=2)
    print("\nResults:", json.dumps(results, indent=2))
    return results


## 6. Smoke test — run this first

Small `replay_start_size`/`sync_target_frames` and a `max_frames` cap so it finishes in
under a minute. This only checks that the environment, wrappers, network, and loss all
run without shape/dtype errors — it will **not** learn anything meaningful in this many
frames, and `solved` will be `False`. Confirm you see `Results: {...}` printed with no
traceback before moving on to the real run below.

In [ ]:
smoke_results = run_training(
    run_tag="smoke_test",
    max_frames=2000,
    replay_start_size=200,
    sync_target_frames=50,
    epsilon_decay_last_frame=1000,
)


## 7. TensorBoard

Launch this before (or during) the real run below so you can watch `reward_100` live.
This is also your source for the "speed of convergence" plot for the report.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs


## 8. Baseline run

This is the actual baseline for Step 1 of the assignment: matches the book's
hyperparameters exactly, stops when the 100-episode mean reward exceeds
`MEAN_REWARD_BOUND` (19.5, same threshold the book uses). On an A100 this should
be noticeably faster than the book's ~10 minutes on a GTX 1080 Ti — but note Atari
DQN is largely **CPU-bound** on environment stepping (not GPU compute), so the A100
mainly helps if you increase batch size/network size later, not necessarily on this
exact baseline config. Record whatever `wall_clock_minutes` and `best_mean_reward`
you get here — that's your Step 1 deliverable.

In [ ]:
baseline_results = run_training(run_tag="baseline")


## 9. Persist results (Colab runtimes are ephemeral)

Saves the TensorBoard logs, best checkpoint, and results JSON to Google Drive so they
survive a runtime disconnect. Mount is commented out by default — uncomment to use it.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/rl-project/baseline
# !cp -r runs baseline-best.dat baseline-results.json /content/drive/MyDrive/rl-project/baseline/
# print("Saved to Google Drive: MyDrive/rl-project/baseline")


## 10. Baseline summary for the report

Fill this straight into the report table: GPU used, wall-clock time to solve,
frames/episodes needed, and best mean reward — compare against the book's claimed
~10 min / mean reward 19 on a GTX 1080 Ti.

In [ ]:
import json
with open("baseline-results.json") as f:
    print(json.dumps(json.load(f), indent=2))


## 11. Evaluate a checkpoint directly (use this if a run was interrupted)

`*-best.dat` only stores network weights &mdash; no reward/frame metadata. If a run was
manually stopped and you don't have a clean `results.json`, load the checkpoint back in
and measure its actual performance with a real (greedy) evaluation instead of digging
through scrollback for a `Best mean reward updated` line.

In [ ]:
def evaluate_checkpoint(checkpoint_path, env_name=DEFAULT_ENV_NAME, n_episodes=25, epsilon=0.0):
    env = wrappers.make_env(env_name)
    net = dqn_model.DQN(env.observation_space.shape, env.action_space.n).to(device)
    net.load_state_dict(torch.load(checkpoint_path, map_location=device))
    net.eval()

    rewards = []
    for ep in range(n_episodes):
        state, _ = env.reset()
        total_reward = 0.0
        done = False
        while not done:
            if np.random.random() < epsilon:
                action = env.action_space.sample()
            else:
                state_v = torch.as_tensor(np.array([state]), dtype=torch.float32).to(device)
                with torch.no_grad():
                    q_vals = net(state_v)
                action = int(torch.argmax(q_vals, dim=1).item())
            state, reward, terminated, truncated, _ = env.step(action)
            total_reward += reward
            done = terminated or truncated
        rewards.append(total_reward)
        print(f"episode {ep + 1}/{n_episodes}: reward={total_reward}")

    env.close()
    mean_r = float(np.mean(rewards))
    std_r = float(np.std(rewards))
    print(f"\nEvaluated {checkpoint_path}: mean reward {mean_r:.3f} +/- {std_r:.3f} over {n_episodes} episodes")
    return rewards, mean_r, std_r


eval_rewards, eval_mean, eval_std = evaluate_checkpoint("baseline-best.dat", n_episodes=25)
